In [ ]:
import os
import torch

from tqdm.auto import tqdm
from model import ContextUnet
from torch.utils.data import DataLoader
from data_utilities import SpritesDataset, transform

import torch.nn.functional as F

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else torch.device("cpu"))
print(f"{device=}")

In [ ]:
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

timesteps = 500

beta1 = 1e-4
beta2 = 0.02

n_epochs = 32

batch_size = 256
learning_rate = 1e-3

In [ ]:
nn_model = ContextUnet(in_channels=3, n_features=64, n_context_features=8, img_dim=16).to(device)

optim = torch.optim.Adam(nn_model.parameters(), lr=learning_rate)

In [ ]:
# construct DDPM noise schedule
b_t = (beta2 - beta1) * torch.linspace(0, 1, timesteps + 1, device=device) + beta1
a_t = 1 - b_t
ab_t = torch.cumsum(a_t.log(), dim=0).exp()
ab_t[0] = 1


In [ ]:
def perturb_input(x, timestep, noise):
    return ab_t.sqrt()[timestep, None, None, None] * x + (1 - ab_t.sqrt()[timestep, None, None, None]) * noise

In [ ]:
dataset = SpritesDataset("/data/sprites/sprites_1788_16x16.npy", "/data/sprites/sprite_labels_nc_1788_16x16.npy", transform=transform)
train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

In [ ]:
nn_model = torch.compile(nn_model)

for ep in range(n_epochs):
    optim.param_groups[0]["lr"] = learning_rate * (1 - ep / n_epochs)
    
    pbar = tqdm(train_dataloader, desc=f"Epoch {ep+1}/{n_epochs}")
    for x, _ in pbar:
        optim.zero_grad()
        x = x.to(device)
        t = torch.randint(1, timesteps + 1, (x.shape[0],), device=device)
        noise = torch.randn_like(x, device=device)
        x_perturbed = perturb_input(x, t, noise)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            predicted_noise = nn_model(x_perturbed, t / timesteps)
            loss = F.mse_loss(predicted_noise, noise)
            loss.backward()

        optim.step()

    if (ep + 1) % 4 == 0 or (ep + 1) == n_epochs:
        torch.save(nn_model.state_dict(), os.path.join(checkpoint_dir, f"ddpm_epoch_{ep+1:03d}.pth"))

